# Week 6 Lab: Connect Preprocessing to Training and Testing

You will split the labeled account data first, then build a Pipeline that learns preprocessing settings and a regression estimator from the training rows. You will predict for held-out test rows and compare those predictions with known targets. A separate future-row example will show how the fitted workflow is used when no target is available.


## Workflow requirements

- Target: `monthly_charges`
- Exclude: `account_id`
- Nominal: `region`, `contact_channel`
- Ordinal: `plan_tier`, `service_priority`
- Numerical feature: `tenure_months`
- Plan order: basic < plus < pro
- Priority order: standard < priority < premium
- Unknown nominal values: ignore during transformation
- Unknown ordinal values: encode as -1
- Numerical transformation: StandardScaler
- Estimator: LinearRegression
- Split the raw X and y before fitting, using `test_size=0.25` and `random_state=42`.
- Fit the Pipeline on `X_train` and `y_train` only.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = Path("data/week6_demo_accounts.csv")
accounts = pd.read_csv(DATA_PATH)
accounts.head()


## Part 1: Separate and split the labeled data

Keep the target in `y`. Do not include the identifier or target in `X`. Split the raw rows before building or fitting the preprocessor. The test rows are held back to represent examples the workflow has not learned from.


In [ ]:
target_column = "monthly_charges"
identifier_column = "account_id"

# TODO: Complete each feature list from the workflow requirements.
nominal_features = []
ordinal_features = []
numeric_features = []

plan_order = ["basic", "plus", "pro"]
priority_order = ["standard", "priority", "premium"]

feature_columns = nominal_features + ordinal_features + numeric_features
assert len(feature_columns) == 5, "Complete the three feature lists first"

# TODO: Create X from feature_columns and y from target_column.
X = accounts[feature_columns]
y = accounts[target_column]

# TODO: Split X and y before fitting any preprocessing step.
# Use test_size=0.25 and random_state=42.
X_train, X_test, y_train, y_test = None, None, None, None


## Part 2: Build the preprocessor

Create one transformer branch per feature group. The transformer rules are specified here, but any learned category list or numerical statistics must come from the training rows when the Pipeline is fitted.


In [ ]:
# TODO: Create a ColumnTransformer named preprocessor.
# Use OneHotEncoder(handle_unknown="ignore", sparse_output=False) for nominal data.
# Use OrdinalEncoder with the supplied orders, unknown_value=-1, and
# handle_unknown="use_encoded_value" for ordinal data.
# Use StandardScaler for numeric data.
# Set remainder="drop" and verbose_feature_names_out=True.
# Finish with .set_output(transform="pandas").
preprocessor = None


## Part 3: Create and fit the Pipeline on training rows

Place the preprocessor first and LinearRegression() second. Fit the complete workflow with `X_train` and `y_train`. The test rows must not be included in the fit call.


In [ ]:
# TODO: Create a Pipeline named workflow with steps named
# "preprocessor" and "model".
workflow = None

# TODO: Fit workflow using X_train and y_train only.


## Part 4: Predict for held-out test rows

Use the original feature columns in `X_test`. The fitted Pipeline applies its saved preprocessing settings, then predicts. Compare the predictions with `y_test`, which stayed out of fitting.


In [ ]:
# TODO: Predict for the held-out X_test rows using the fitted workflow.
test_predictions = None

test_results = X_test.copy()
test_results.insert(0, f"actual_{target_column}", y_test)
test_results[f"predicted_{target_column}"] = test_predictions
test_results


## Part 5: Predict future rows and verify

These additional example rows have no target value. Use the same fitted Pipeline to predict for them; do not fit the workflow again. Then complete the checks to distinguish the held-out test subset from future rows.


In [ ]:
future_accounts = pd.DataFrame(
    {
        "region": ["east", "central"],
        "contact_channel": ["chat", "satellite"],
        "plan_tier": ["pro", "enterprise"],
        "service_priority": ["premium", "urgent"],
        "tenure_months": [30, 70],
    }
)

# TODO: Use the fitted workflow to predict the future_accounts rows. Do not fit again.
future_predictions = None
future_results = future_accounts.copy()
future_results[f"predicted_{target_column}"] = future_predictions

fitted_preprocessor = workflow.named_steps["preprocessor"]
feature_names = list(fitted_preprocessor.get_feature_names_out())
X_train_prepared = fitted_preprocessor.transform(X_train)
X_test_prepared = fitted_preprocessor.transform(X_test)
future_prepared = fitted_preprocessor.transform(future_accounts)

# TODO: Add assertions to verify:
# - train and test indices are disjoint and together cover the original X rows.
# - the Pipeline steps are named "preprocessor" and "model".
# - the identifier and target are not included in X.
# - transformed training, test, and future rows have the expected shapes.
# - test and future prediction counts align with their input rows.
# - all predictions are finite.
# - the unseen ordinal label in future_accounts becomes -1.
# Hint: Use feature_names to locate ordinal__service_priority.
# Finish with print("All Week 6 lab checks passed.")


## Short response

In your own words, explain:

1. Why do we split the original rows before fitting preprocessing or the estimator?
2. What does the preprocessor learn from `X_train`, and what does the estimator learn from the transformed training rows and `y_train`?
3. Why can we compare test predictions with `y_test`, but not score predictions for `future_accounts` yet?
4. What happens when you call `workflow.predict(X_test)`?
5. What does the -1 value tell you about the unseen ordinal label, and what does it not tell you?
6. The dataset has 18 rows and the split uses `test_size=0.25`. Why is the test set not exactly 25% of the rows, and how many rows end up in the training and test sets?
